# Lesson 07: Building Text Generation Applications

In this notebook, you will build 3 complete Python apps using `google-genai`:
1. **2-Stage Recipe & Shopping List Chainer** (with regex input validation).
2. **Historical Persona Bot** (Abraham Lincoln with timeline grounding).
3. **Python Study Buddy** (`- Concept`, `- Example code`, `- Explanation`).

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os, re
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
def sanitize_text(text):
    if not re.fullmatch(r"[a-zA-Z0-9\s,.\-']+", text.strip()):
        raise ValueError("Invalid characters in input")
    return text.strip()

ingredients = sanitize_text("chicken, broccoli, carrots")
dietary_filter = sanitize_text("no peanuts")

if client:
    p1 = f"Show me 2 quick recipes using {ingredients}. Dietary rule: {dietary_filter}. Include nutrition per serving."
    r1 = client.models.generate_content(model="gemini-2.5-flash", contents=p1, config=types.GenerateContentConfig(temperature=0.7))
    print("=== STAGE 1: RECIPES ===\n", r1.text[:500])
    p2 = f"{r1.text}\n\nProduce a concise shopping list for these recipes, excluding ingredients I already have: {ingredients}."
    r2 = client.models.generate_content(model="gemini-2.5-flash", contents=p2, config=types.GenerateContentConfig(temperature=0.2))
    print("\n=== STAGE 2: SHOPPING LIST ===\n", r2.text)
else:
    print("[Offline Demo Mode] Stage 1: Chicken & Broccoli Stir-Fry. Stage 2 Shopping List: garlic, olive oil, soy sauce.")

In [ ]:
if client:
    hb = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="What do you think about smartphones?",
        config=types.GenerateContentConfig(
            temperature=0.0,
            system_instruction="You are Abraham Lincoln. Only answer based on your historical lifetime (1809-1865); if asked about later inventions, state you have no knowledge of them.",
        ),
    )
    print("History Bot:\n", hb.text)
else:
    print("[Offline Demo Mode] History Bot: I am unfamiliar with 'smartphones', as my lifetime ended in 1865.")